# 05 · Escenarios de Concept Drift

Objetivo (PDF sección 3.5): simular los escenarios de política de olvido definidos en el checkpoint 1, es decir sin olvido (todo el historial), el escenario A (ventana deslizante fija, con distintos tamaños) y el escenario B (time-decay ponderado), y comparar su impacto real en el rendimiento a lo largo del tiempo.

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from models import get_models
from evaluation import compute_metrics, business_cost
from drift import time_decay_weights

train_df = pd.read_parquet('../data/processed/train.parquet')
val_df = pd.read_parquet('../data/processed/val.parquet')
test_df = pd.read_parquet('../data/processed/test.parquet')
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True).sort_values('TransactionDT').reset_index(drop=True)

drop_cols = ['TransactionID', 'isFraud', 'TransactionDT', 'day']
feature_cols = [c for c in full_df.columns if c not in drop_cols]

THRESHOLD = 0.07  # el mismo umbral costo-optimo del notebook 03, no se re-optimiza aqui
print('full_df:', full_df.shape, ' dias:', full_df['day'].min(), '-', full_df['day'].max())

full_df: (590540, 401)  dias: 0 - 181


Se combinan train, val y test en una sola línea de tiempo continua. Para comparar políticas de olvido necesitamos reentrenar en distintos puntos de corte y evaluar hacia adelante, algo que el split fijo train/val/test de los notebooks 02 a 04 no permite por sí solo. El umbral (0.07) se mantiene fijo en todos los escenarios para que la comparación sea justa, no se vuelve a optimizar por escenario. Es una limitación consciente y no un descuido.

## 1. Diseño del experimento

Se eligen 2 puntos de corte, día 120 y día 150, cada uno seguido de un bloque de evaluación de 20 días. Para cada corte se entrena XGBoost con 5 estrategias distintas usando solo datos anteriores al corte, y se evalúa sobre el mismo bloque futuro:

- Sin olvido: todo el historial disponible, con pesos uniformes.
- Ventana de 30, 60 o 90 días (escenario A): solo los últimos N días antes del corte.
- Time-decay con vida media de 30 días (escenario B): todo el historial, pero con drift.time_decay_weights, donde las transacciones recientes pesan más que las antiguas.

In [1]:
def make_strategy_frames(cutoff_day, eval_days=20):
    history = full_df[full_df['day'] < cutoff_day]
    eval_block = full_df[(full_df['day'] >= cutoff_day) & (full_df['day'] < cutoff_day + eval_days)]
    return history, eval_block

def train_and_eval(history, eval_block, sample_weight=None):
    model = get_models()['xgboost']
    model.fit(history[feature_cols], history['isFraud'], sample_weight=sample_weight)
    proba = model.predict_proba(eval_block[feature_cols])[:, 1]
    pred = (proba >= THRESHOLD).astype(int)
    m = compute_metrics(eval_block['isFraud'], pred, proba)
    m['cost'] = business_cost(eval_block['isFraud'], pred)
    m['n_train'] = len(history)
    m['n_eval'] = len(eval_block)
    return m

cutoffs = [120, 150]
window_sizes = [30, 60, 90]
results = []

for cutoff in cutoffs:
    history, eval_block = make_strategy_frames(cutoff)

    m = train_and_eval(history, eval_block)
    m.update(cutoff=cutoff, strategy='sin_olvido')
    results.append(m)

    for w in window_sizes:
        window_history = history[history['day'] >= cutoff - w]
        m = train_and_eval(window_history, eval_block)
        m.update(cutoff=cutoff, strategy=f'ventana_{w}d')
        results.append(m)

    weights = time_decay_weights(history['TransactionDT'], half_life=30 * 86400)
    m = train_and_eval(history, eval_block, sample_weight=weights)
    m.update(cutoff=cutoff, strategy='time_decay_30d')
    results.append(m)

    print(f'corte={cutoff} listo')

results_df = pd.DataFrame(results)[['cutoff', 'strategy', 'n_train', 'n_eval', 'f1', 'roc_auc', 'cost']]
print()
print(results_df.to_string(index=False))

corte=120 listo
corte=150 listo

 cutoff       strategy  n_train  n_eval       f1  roc_auc     cost
    120     sin_olvido   414542   57464 0.485598 0.920568 108345.0
    120    ventana_30d    98615   57464 0.471843 0.914567 112605.0
    120    ventana_60d   190804   57464 0.484828 0.925102 105555.0
    120    ventana_90d   280203   57464 0.463890 0.923727 111825.0
    120 time_decay_30d   414542   57464 0.486691 0.924623 105000.0
    150     sin_olvido   498113   60334 0.418634 0.908440 123090.0
    150    ventana_30d    83571   60334 0.466289 0.908427 117735.0
    150    ventana_60d   182186   60334 0.435409 0.913671 118005.0
    150    ventana_90d   274375   60334 0.419945 0.912266 120375.0
    150 time_decay_30d   498113   60334 0.426437 0.914365 118425.0


En los dos cortes, la estrategia sin olvido (todo el historial) nunca es la mejor, y en el corte 150 es directamente la peor ($123,090 contra $117,735 de la ventana de 30 días). Esto confirma la premisa del checkpoint 1 de priorizar estrategias de olvido por encima de reentrenar con todo el histórico.

Pero ninguna ventana gana en los dos cortes a la vez. En el corte 120 casi empatan la ventana de 60 días y el time-decay ($105,555 y $105,000), mientras que en el corte 150 gana claramente la ventana de 30 días ($117,735). Esto sugiere que el tamaño de ventana óptimo también cambia con el tiempo, algo coherente con que el propio concept drift no es estacionario. Las diferencias son modestas en magnitud, entre 3% y 4.3% de ahorro en costo, pero consistentes en dirección: olvidar algo del pasado ayuda, aunque no hay un tamaño de ventana único que sea siempre el mejor.

## 2. Comparación visual

In [1]:
strategies = ['sin_olvido', 'ventana_30d', 'ventana_60d', 'ventana_90d', 'time_decay_30d']
labels = ['Sin olvido', 'Ventana 30d', 'Ventana 60d', 'Ventana 90d', 'Time-decay 30d']
colors = ['#64748b', '#0284c7', '#0369a1', '#0f172a', '#16a34a']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
width = 0.15
x = np.arange(len(cutoffs))

for i, strat in enumerate(strategies):
    costs = [results_df[(results_df['cutoff'] == c) & (results_df['strategy'] == strat)]['cost'].iloc[0] for c in cutoffs]
    axes[0].bar(x + i * width, costs, width, label=labels[i], color=colors[i])
    f1s = [results_df[(results_df['cutoff'] == c) & (results_df['strategy'] == strat)]['f1'].iloc[0] for c in cutoffs]
    axes[1].bar(x + i * width, f1s, width, label=labels[i], color=colors[i])

for ax, title, ylabel in zip(axes, ['Costo de negocio por estrategia', 'F1 por estrategia'], ['Costo ($)', 'F1']):
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels([f'corte dia {c}' for c in cutoffs])
    ax.set_title(title)
    ax.set_ylabel(ylabel)

axes[0].legend(fontsize=8, loc='upper left', bbox_to_anchor=(0, -0.15), ncol=3)
plt.tight_layout()
plt.savefig('../figures/08_drift_scenarios_comparison.png', dpi=110, bbox_inches='tight')
plt.show()

![Comparación de escenarios de drift](../figures/08_drift_scenarios_comparison.png)

## 3. Reflexión crítica sobre riesgos (PDF sección 3.5)

- El hallazgo de que el tamaño de ventana óptimo cambia entre cortes es en sí mismo un riesgo de confianza y estabilidad. Un sistema que fija un solo tamaño de ventana para siempre puede quedar mal calibrado cuando el patrón de drift cambia de naturaleza. Automatizar también la elección del tamaño de ventana agrega una capa más de complejidad y de posible fragilidad, no necesariamente una solución.
- Sobre la toma de decisiones automatizada, ninguna de estas estrategias elimina la necesidad de la restricción de decisión definida en el checkpoint 1: los casos de score ambiguo se siguen derivando a un analista humano. La adaptación automática del modelo no debería traducirse en menos supervisión humana sobre las decisiones individuales.
- Hay también un riesgo de retroalimentación. Si el reentrenamiento futuro usa las propias decisiones del sistema, en vez de fraude confirmado de forma independiente, como etiqueta, un error sistemático del modelo podría reforzarse a sí mismo con el tiempo. La política de reentrenamiento debe basarse en fraude confirmado, no en las predicciones del modelo.
- Por último está el costo operativo de reentrenar seguido. Las ventanas más cortas, como la de 30 días, implican reentrenar con más frecuencia y con menos datos por corrida, un trade-off entre adaptabilidad y estabilidad estadística del modelo que hay que monitorear y no dar por resuelto.

## Conclusiones finales del pipeline (notebooks 01 a 05)

- El EDA del notebook 01 encontró drift real (tasa de fraude no estacionaria, cambio de mix de ProductCD), y no era solo una hipótesis del checkpoint 1.
- El preprocesamiento del notebook 02 generó un split cronológico limpio, sin fuga de información.
- El modelado del notebook 03 mostró que XGBoost con un umbral optimizado por costo (0.07) supera claramente el umbral por defecto, con un 28.3% de ahorro en val.
- La evaluación temporal del notebook 04 confirmó una degradación real entre val y test (18% de caída en F1) que superaría el umbral de alerta del checkpoint 1, aunque sin colapso monótono dentro de la ventana de test.
- Los escenarios de olvido del notebook 05 confirman que alguna forma de olvido siempre ayuda frente a usar todo el historial, aunque el tamaño de ventana óptimo no es fijo, lo que refuerza la necesidad de monitoreo continuo en vez de una política de reentrenamiento estática.